In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import timm, types, sys
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from peft import LoraConfig
from peft.tuners.lora import LoraModel
from sklearn.metrics import f1_score
import torchvision.transforms as T

sys.path.append(".")
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
DATASET_NAME = "BREAKHIS"   # cambia qui per altri dataset

CFG = dict(
    data_dir        = f"/data/{DATASET_NAME}",
    img_size        = 224,
    batch_size      = 64,
    num_workers     = 8,
    seed            = 42,
    classifier_ckpt = Path("checkpoints/uni_finetuned/best_model.pt"),
    forecaster_ckpt = Path("checkpoints/forecaster/forecaster_src05_tgt23.pt"),
    finetuned_pruned_ckpt = Path("checkpoints/pruned_finetuned/best_pruned_layer5_keep30.pt"),
    results_dir     = Path(f"results/{DATASET_NAME}"),
    prune_layer     = 5,
    far_threshold   = 1e-4,
    keep_ratios     = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
CFG["results_dir"].mkdir(parents=True, exist_ok=True)


In [3]:
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])
test_ds     = HistologicalImageDataset(f"{CFG['data_dir']}/test", transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"],
                         shuffle=False, num_workers=CFG["num_workers"],
                         pin_memory=True, persistent_workers=True)
CLASS_NAMES = test_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Test samples: {len(test_ds)} | Classi: {CLASS_NAMES}")


Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Test samples: 6833 | Classi: ['adenosis', 'fibroadenoma', 'phyllodes_tumor', 'tubular_adenoma', 'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']


In [4]:
def compute_tar_at_far(scores, is_correct, far_threshold=1e-4):
    scores    = np.array(scores)
    correct   = np.array(is_correct).astype(bool)
    incorrect = ~correct
    if incorrect.sum() == 0:
        return 1.0
    n_far     = max(1, int(np.ceil(incorrect.sum() * far_threshold)))
    threshold = np.sort(scores[incorrect])[::-1][min(n_far-1, incorrect.sum()-1)]
    return float((scores[correct] >= threshold).mean())


def compute_metrics(all_preds, all_labels, all_scores, far_threshold=1e-4):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    f1         = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    is_correct = (all_preds == all_labels)
    tar        = compute_tar_at_far(all_scores, is_correct, far_threshold)
    return {"f1_macro": f1, "tar_at_far": tar}


In [5]:
class UNILoRAClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.1):
        super().__init__()
        backbone = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True,
                                      init_values=1e-5, dynamic_img_size=True)
        lora_config = LoraConfig(r=8, lora_alpha=32,
            target_modules=["qkv","proj","fc1","fc2"],
            lora_dropout=0.1, bias="none")
        self.backbone = LoraModel(backbone, lora_config, adapter_name="default")
        self.head = nn.Sequential(
            nn.LayerNorm(1024), nn.Dropout(dropout), nn.Linear(1024, n_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


class AttentionForecaster(nn.Module):
    def __init__(self, embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(embed_dim, hidden)
        self.cls_query   = nn.Parameter(torch.randn(1,1,hidden)*0.02)
        self.self_attn   = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=hidden, nhead=n_heads,
                dim_feedforward=hidden*2, dropout=dropout,
                batch_first=True, norm_first=True) for _ in range(n_layers)])
        self.cross_attn  = nn.ModuleList([
            nn.MultiheadAttention(hidden, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)])
        self.cross_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.norm        = nn.LayerNorm(hidden)
        self.score_head  = nn.Sequential(
            nn.Linear(hidden*2,128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128,1))
    def forward(self, x):
        B, N, D = x.shape
        x = self.input_proj(x)
        for sa in self.self_attn: x = sa(x)
        cls = self.cls_query.expand(B,-1,-1)
        for ca, norm in zip(self.cross_attn, self.cross_norms):
            cls_out, _ = ca(cls, x, x); cls = norm(cls + cls_out)
        x_norm  = self.norm(x)
        cls_exp = cls.expand(-1,N,-1)
        return self.score_head(torch.cat([x_norm,cls_exp],dim=-1)).squeeze(-1).softmax(-1)


# Carica classificatore fine-tuned
classifier = UNILoRAClassifier(N_CLASSES).to(device)
classifier.load_state_dict(
    torch.load(CFG["classifier_ckpt"], map_location=device), strict=False)
classifier.eval()
for p in classifier.parameters(): p.requires_grad_(False)

# Carica forecaster
forecaster = AttentionForecaster().to(device)
forecaster.load_state_dict(torch.load(CFG["forecaster_ckpt"], map_location=device))
forecaster.eval()
for p in forecaster.parameters(): p.requires_grad_(False)

# Carica classificatore fine-tuned CON pruning
classifier_ft_pruned = UNILoRAClassifier(N_CLASSES).to(device)
classifier_ft_pruned.load_state_dict(
    torch.load(CFG["finetuned_pruned_ckpt"], map_location=device), strict=False)
classifier_ft_pruned.eval()
for p in classifier_ft_pruned.parameters(): p.requires_grad_(False)

print("Modelli caricati")

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/uni_finetuned/best_model.pt'

In [ ]:
def forward_with_pruning(model, imgs, device, prune_layer, keep_ratio, score_fn):
    """
    score_fn: callable(patch_emb [B,196,D], attn [B,H,N,N]) → scores [B,196]
    Esegue pruning fisico dopo prune_layer.
    """
    cache = {}
    orig_blocks = {}

    def make_block_hook(idx):
        orig_fwd = model.backbone.model.blocks[idx].forward
        def block_fwd(x):
            # Hook separato sull'attention per catturare attn weights
            attn_cache = {}
            orig_attn  = model.backbone.model.blocks[idx].attn.forward

            def attn_hook(self_attn, x_in):
                B, N, C = x_in.shape
                qkv  = self_attn.qkv(x_in).reshape(B,N,3,self_attn.num_heads,
                         self_attn.head_dim).permute(2,0,3,1,4)
                q, k, v = qkv.unbind(0)
                q, k   = self_attn.q_norm(q), self_attn.k_norm(k)
                attn   = (q @ k.transpose(-2,-1) * self_attn.scale).softmax(-1)
                attn_cache["attn"] = attn.detach()
                attn_cache["emb"]  = x_in[:, 1:].detach()
                x_out = (self_attn.attn_drop(attn) @ v).transpose(1,2).reshape(B,N,C)
                return self_attn.proj_drop(self_attn.proj(x_out))

            if idx == prune_layer:
                model.backbone.model.blocks[idx].attn.forward = \
                    types.MethodType(attn_hook, model.backbone.model.blocks[idx].attn)

            x = orig_fwd(x)

            if idx == prune_layer:
                model.backbone.model.blocks[idx].attn.forward = orig_attn
                B, N, D = x.shape
                k_keep  = max(1, int((N-1) * keep_ratio))
                with torch.no_grad():
                    scores = score_fn(attn_cache.get("emb"),
                                      attn_cache.get("attn"))  # [B, 196]
                topk    = scores.topk(k_keep, dim=-1).indices
                cls_tok = x[:, :1, :]
                kept    = torch.stack([x[b,1:][topk[b]] for b in range(B)])
                x = torch.cat([cls_tok, kept], dim=1)
            return x
        return block_fwd

    for i, block in enumerate(model.backbone.model.blocks):
        orig_blocks[i] = block.forward
        block.forward  = make_block_hook(i)

    with torch.no_grad():
        logits = model(imgs.to(device))

    for i, block in enumerate(model.backbone.model.blocks):
        block.forward = orig_blocks[i]

    return logits


In [ ]:
def score_random(emb, attn):
    """Selezione casuale."""
    B, N, D = emb.shape
    return torch.rand(B, N, device=emb.device)


def score_cls_attention(emb, attn):
    """CLS attention del layer corrente."""
    # attn: [B, H, N+1, N+1]
    return attn[:, :, 0, 1:].mean(1)   # [B, 196]


def score_token_norm(emb, attn):
    """Norma L2 dei token embedding."""
    return emb.norm(dim=-1)             # [B, 196]


def score_attention_entropy(emb, attn):
    """Patch con attenzione poco focalizzata = bassa importanza.
    Score = entropia negativa (bassa entropia = focalizzata = importante)."""
    patch_attn = attn[:, :, 1:, 1:]    # [B, H, 196, 196]
    ent = -(patch_attn * (patch_attn + 1e-8).log()).sum(-1).mean(1)  # [B, 196]
    return -ent                         # inverti


def score_center_bias(emb, attn):
    """Prior spaziale: patch centrali più importanti.
    Gaussiana centrata sull'immagine 14x14."""
    B   = emb.shape[0]
    idx = torch.arange(196, device=emb.device)
    r   = (idx // 14).float() - 6.5
    c   = (idx %  14).float() - 6.5
    dist = (r**2 + c**2).sqrt()
    score = (-dist / 4.0).exp()         # gaussiana σ=4
    return score.unsqueeze(0).expand(B, -1)


def score_patch_embed_norm(emb, attn):
    """Norma dell'embedding nel primo layer (proxy attivazione iniziale).
    Nota: emb qui è al prune_layer, non al layer 0 — è comunque un proxy."""
    return emb.norm(dim=-1)


def make_score_forecaster(forecaster):
    """Forecaster (frozen) — predice importanza futura."""
    def score_fn(emb, attn):
        return forecaster(emb)
    return score_fn

In [ ]:
METHODS = [
    # (nome, score_fn, modello)
    ("Random",
     score_random,
     classifier),

    ("CLS Attention",
     score_cls_attention,
     classifier),

    ("Token Norm",
     score_token_norm,
     classifier),

    ("Attention Entropy",
     score_attention_entropy,
     classifier),

    ("Center Bias",
     score_center_bias,
     classifier),

    ("Forecaster (no FT)",
     make_score_forecaster(forecaster),
     classifier),          # classificatore NON fine-tuned per il pruning

    ("Forecaster + FT",
     make_score_forecaster(forecaster),
     classifier_ft_pruned),  # classificatore fine-tuned CON pruning
]


In [ ]:
results = []  # lista di dict

for method_name, score_fn, model in tqdm(METHODS, desc="Methods"):
    for keep_ratio in tqdm(CFG["keep_ratios"], desc=method_name, leave=False):
        all_preds, all_labels, all_scores = [], [], []

        for imgs, labels in test_loader:
            logits = forward_with_pruning(
                model, imgs, device,
                prune_layer = CFG["prune_layer"],
                keep_ratio  = keep_ratio,
                score_fn    = score_fn,
            )
            probs  = logits.softmax(-1).cpu()
            preds  = probs.argmax(-1)
            scores = probs.max(-1).values
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
            all_scores.extend(scores.numpy())

        metrics = compute_metrics(all_preds, all_labels, all_scores,
                                   CFG["far_threshold"])
        results.append({
            "method"      : method_name,
            "keep_ratio"  : keep_ratio,
            "keep_pct"    : int(keep_ratio * 100),
            "f1_macro"    : metrics["f1_macro"],
            "tar_at_far"  : metrics["tar_at_far"],
        })
        print(f"  {method_name:25s} keep={int(keep_ratio*100):3d}% "
              f"| F1={metrics['f1_macro']:.3f} "
              f"| TAR={metrics['tar_at_far']:.3f}")


In [ ]:
df = pd.DataFrame(results)
csv_path = CFG["results_dir"] / "pruning_comparison.csv"
df.to_csv(csv_path, index=False)
print(f"Risultati salvati in {csv_path}")
print(df.to_string(index=False))

In [ ]:
METHOD_STYLES = {
    "Random"              : {"color": "#aaaaaa", "ls": "--",  "marker": "x",  "lw": 1.5},
    "CLS Attention"       : {"color": "#4878cf", "ls": "--",  "marker": "s",  "lw": 1.5},
    "Token Norm"          : {"color": "#6acc65", "ls": "--",  "marker": "^",  "lw": 1.5},
    "Attention Entropy"   : {"color": "#d65f5f", "ls": "--",  "marker": "v",  "lw": 1.5},
    "Center Bias"         : {"color": "#b47cc7", "ls": "--",  "marker": "D",  "lw": 1.5},
    "Forecaster (no FT)"  : {"color": "#ee8866", "ls": "-",   "marker": "o",  "lw": 2.0},
    "Forecaster + FT"     : {"color": "#e63946", "ls": "-",   "marker": "o",  "lw": 2.5},
}

x_vals = [int(kr*100) for kr in CFG["keep_ratios"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.subplots_adjust(wspace=0.35)

for metric, ylabel, ax in [
    ("f1_macro",  "F1 Macro",             axes[0]),
    ("tar_at_far", f"TAR @ FAR={CFG['far_threshold']:.0e}", axes[1]),
]:
    for method_name, style in METHOD_STYLES.items():
        sub = df[df["method"] == method_name].sort_values("keep_pct")
        if sub.empty:
            continue
        ax.plot(sub["keep_pct"], sub[metric],
                color=style["color"], ls=style["ls"],
                marker=style["marker"], lw=style["lw"],
                label=method_name, markersize=5)

    ax.set_xlabel("Keep ratio (%)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(x_vals)
    ax.tick_params(labelsize=10)
    ax.grid(alpha=0.25, linestyle=":")
    ax.spines[["top","right"]].set_visible(False)

axes[0].legend(fontsize=8, loc="lower right", framealpha=0.9,
               ncol=1, handlelength=2)
axes[1].legend(fontsize=8, loc="lower right", framealpha=0.9,
               ncol=1, handlelength=2)

fig.suptitle(f"Patch Pruning — {DATASET_NAME}\n"
             f"Prune layer: {CFG['prune_layer']}",
             fontsize=13, y=1.02)

plot_path = CFG["results_dir"] / "pruning_comparison.pdf"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
fig.savefig(str(plot_path).replace(".pdf",".png"), dpi=300, bbox_inches="tight")
print(f"Plot salvato in {plot_path}")
plt.show()

In [ ]:
pivot_f1  = df.pivot(index="method", columns="keep_pct", values="f1_macro")
pivot_tar = df.pivot(index="method", columns="keep_pct", values="tar_at_far")

def to_latex_table(pivot, metric_name, dataset_name):
    cols = pivot.columns.tolist()
    header = " & ".join([f"{c}\\%" for c in cols])
    lines  = [
        f"\\begin{{table}}[h]",
        f"\\caption{{{metric_name} — {dataset_name}}}",
        f"\\begin{{tabular}}{{l{'c'*len(cols)}}}",
        f"\\toprule",
        f"Method & {header} \\\\",
        f"\\midrule",
    ]
    for method, row in pivot.iterrows():
        best_col = row.idxmax()
        vals = []
        for col in cols:
            v = row[col]
            if col == best_col:
                vals.append(f"\\textbf{{{v:.3f}}}")
            else:
                vals.append(f"{v:.3f}")
        lines.append(f"{method} & {' & '.join(vals)} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)

latex_f1  = to_latex_table(pivot_f1,  "F1 Macro",  DATASET_NAME)
latex_tar = to_latex_table(pivot_tar, "TAR@FAR",   DATASET_NAME)

latex_path_f1  = CFG["results_dir"] / "table_f1.tex"
latex_path_tar = CFG["results_dir"] / "table_tar.tex"
latex_path_f1.write_text(latex_f1)
latex_path_tar.write_text(latex_tar)
print(f"Tabelle LaTeX salvate in {CFG['results_dir']}")
print("\n── F1 Macro ──")
print(latex_f1)